# RAG01｜機器學習筆記知識庫
用 ChatGPT 協助將第五週筆記整理成 31 個主題區塊，以 JSON 保存正文與來源頁碼。
本機 RTX 5060 負責向量化，FAISS 負責搜尋；完成後匯出 ZIP 給 RAG02 使用。

## 1. 安裝套件
安裝 Sentence Transformers、Transformers、FAISS 與模型處理器需要的 Torchvision。
本機環境已裝好套件；重裝或換環境時再執行，若已載入舊版本則重啟工作階段。

In [ ]:
%pip install -q "sentence-transformers==6.1.0" "transformers==5.20.1" "faiss-cpu==1.15.1" "torchvision==0.26.0" "ipywidgets>=8,<9"

## 2. 匯入套件與設定
設定模型、每批處理量與匯出檔名，並確認目前使用的 GPU 和套件版本。
支援時使用 bfloat16，否則使用 float32；不使用 float16。

In [4]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import importlib.metadata as package_metadata
import hashlib
import json
import re
import tempfile
import zipfile

import torch
import faiss
from IPython.display import display
from sentence_transformers import SentenceTransformer
from huggingface_hub import HfApi

C:\Users\LeePin\Documents\Codex\2026-10-30\referenced-chatgpt-conversation-this-is-an\work\rag-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1031 17:35:43.693000 55460 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [ ]:
MODEL_ID = "google/embeddinggemma-2"
BATCH_SIZE = 4

# 本次資料與輸出設定
EXPECTED_DOCUMENT_COUNT = 31  # 本次知識庫共有 31 個區塊
EXPORT_ZIP_NAME = "ml_week5_vectorDB.zip"

# text 本身已含標題；這裡不再從 metadata 添加標題。
DOCUMENT_TEMPLATE = "title: none | text: {text}"
QUERY_TEMPLATE = "task: search result | query: {question}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = (
    torch.bfloat16
    if DEVICE == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float32
)
PACKAGE_NAMES = ["torch", "torchvision", "numpy", "sentence-transformers", "transformers", "faiss-cpu", "huggingface-hub"]
PACKAGE_VERSIONS = {name: package_metadata.version(name) for name in PACKAGE_NAMES}
print("裝置：", DEVICE, "／模型精度：", DTYPE)
if DEVICE == "cuda":
    print("GPU：", torch.cuda.get_device_name(0))
display(pd.DataFrame(PACKAGE_VERSIONS.items(), columns=["套件", "版本"]))


## 3. 讀取 JSON
JSON 已放入 31 個區塊，每筆包含 `doc_id`、`text` 與 `metadata`，不用再拆 TXT 或解壓縮。
讀取後檢查正文、來源欄位與編號，後續直接用 `text` 建立向量。

In [ ]:
from pathlib import Path
import json
import hashlib

# 改成你實際存放 JSON 的路徑
json_path = Path(r"C:\Users\LeePin\Desktop\個人\碩士班\生成式AI課\作業七\documents.json")

source_bytes = json_path.read_bytes()
documents = json.loads(source_bytes.decode("utf-8-sig"))

# 檢查正文、來源資訊與區塊編號
if not isinstance(documents, list) or not documents:
    raise ValueError("JSON 必須是非空的區塊清單。")

for doc in documents:
    assert isinstance(doc["doc_id"], str)
    assert isinstance(doc["text"], str) and doc["text"].strip()
    assert isinstance(doc["metadata"], dict)

index_doc_ids = [doc["doc_id"] for doc in documents]
assert len(index_doc_ids) == len(set(index_doc_ids)), "區塊編號重複。"
if EXPECTED_DOCUMENT_COUNT is not None:
    assert len(documents) == EXPECTED_DOCUMENT_COUNT, "區塊數量與預期不同。"

# 保留後續匯出程式需要的變數
archive_name = json_path.name
SOURCE_ARCHIVE_SHA256 = hashlib.sha256(source_bytes).hexdigest()

print("載入檔案：", json_path.name)
print("區塊數量：", len(documents))
print("\n第一塊資料：")
print(json.dumps(documents[0], ensure_ascii=False, indent=2))

## 4. 載入 EmbeddingGemma 2
只載入文字模型，並記錄確切版本，讓 RAG02 使用相同設定。
首次需下載；本機已有快取時會重用檔案，這一步還沒有計算文件向量。

In [ ]:
try:
    MODEL_REVISION = HfApi().model_info(MODEL_ID, token=False).sha
    model = SentenceTransformer(
        MODEL_ID,
        revision=MODEL_REVISION,
        token=False,
        device=DEVICE,
        config_kwargs={"vision_config": None, "audio_config": None},
        model_kwargs={"dtype": DTYPE, "attn_implementation": "eager"},
    )
except Exception as exc:
    raise RuntimeError(
        "EmbeddingGemma 2 載入失敗。請確認網路、套件版本，必要時重新啟動工作階段；"
        "若 GPU 記憶體不足，改用 CPU。未替換成其他模型。"
    ) from exc

print("模型：", MODEL_ID)
print("版本：", MODEL_REVISION)
print("裝置：", model.device)
print("模型精度：", model.dtype)
print("向量維度：", model.get_embedding_dimension())
print("模型單筆 token 上限：", model.max_seq_length)

## 5. 檢查文字長度
文件與問題各加上對應的檢索前綴，先計算實際輸入的 token 數。
本次上限設為 2048；超過就停止處理，避免正文被悄悄截斷。

In [ ]:
document_inputs = [DOCUMENT_TEMPLATE.format(text=doc["text"]) for doc in documents]
TOKEN_LIMIT = min(2048, int(model.max_seq_length))

if TOKEN_LIMIT <= 0:
    raise ValueError("輸入長度上限必須大於0。")

# 將實際 encode 上限與本次檢查預算保持一致。
model.max_seq_length = TOKEN_LIMIT

def count_input_tokens(text):
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])

token_counts = [count_input_tokens(text) for text in document_inputs]
length_table = pd.DataFrame({"doc_id": index_doc_ids, "tokens_含前綴": token_counts})
display(length_table)
long_docs = [doc_id for doc_id, length in zip(index_doc_ids, token_counts) if length > TOKEN_LIMIT]
if long_docs:
    raise ValueError(f"以下文件超過 {TOKEN_LIMIT} tokens，請先調整資料：{long_docs}")
print("本教材使用的 token 上限：", TOKEN_LIMIT)
print("全部文件均未超過上限；不需要截斷。")
print("第一筆真正送入 embedding 的文字：\n", document_inputs[0])

## 6. 產生文件向量
每個區塊產生一個 768 維向量，31 個區塊形成 `(31, 768)` 矩陣。
將向量正規化並轉為 float32，供 FAISS 比較語意相似度。

In [ ]:
embeddings = model.encode(
    document_inputs,
    prompt="",
    batch_size=BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
if embeddings.ndim != 2 or embeddings.shape[0] != len(documents):
    raise ValueError("向量形狀與文件數不符。")
if not np.isfinite(embeddings).all():
    raise ValueError("向量含有 NaN 或無限值。")
if np.any(np.linalg.norm(embeddings, axis=1) <= 0):
    raise ValueError("出現零向量。")
faiss.normalize_L2(embeddings)
DIMENSION = int(embeddings.shape[1])
if DIMENSION != int(model.get_embedding_dimension()):
    raise ValueError("實際向量維度與模型宣告不符。")
print("矩陣形狀（文件數, 向量維度）：", embeddings.shape)
print("第一篇向量的前10個數字：", embeddings[0, :10])
print("前5篇向量長度：", np.linalg.norm(embeddings[:5], axis=1))

## 7. 建立 FAISS 索引
使用 `IndexFlatIP` 搜尋正規化向量，內積就是 cosine similarity。
FAISS 回傳向量位置，再透過 `index_doc_ids` 找回正文與來源。

In [11]:
index = faiss.IndexFlatIP(DIMENSION)
index.add(embeddings)
documents_by_id = {doc["doc_id"]: doc for doc in documents}
assert index.ntotal == len(index_doc_ids) == len(documents)
print("索引類型：", type(index).__name__)
print("索引向量數：", index.ntotal)
print("索引維度：", index.d)
display(pd.DataFrame({"FAISS編號": range(len(index_doc_ids)), "doc_id": index_doc_ids}))

索引類型： IndexFlatIP
索引向量數： 31
索引維度： 768


,FAISS編號,doc_id
0,0,doc_001
1,1,doc_002
2,2,doc_003
3,3,doc_004
4,4,doc_005
5,5,doc_006
6,6,doc_007
7,7,doc_008
8,8,doc_009
9,9,doc_010


## 8. 定義搜尋函式
問題使用同一個 embedding 模型轉成向量，再找出最相近的 top-k 個區塊。
這裡只找資料，還沒呼叫 LLM；沒有答案的問題也可能搜到相近內容。

In [12]:
def embed_question(question):
    if not isinstance(question, str) or not question.strip():
        raise ValueError("請輸入非空白問題。")
    query_input = QUERY_TEMPLATE.format(question=question.strip())
    if count_input_tokens(query_input) > TOKEN_LIMIT:
        raise ValueError("問題超過模型輸入長度上限，請縮短問題。")
    query_vector = model.encode(
        [query_input], prompt="", convert_to_numpy=True,
        normalize_embeddings=True, show_progress_bar=False,
    )
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    if query_vector.shape != (1, DIMENSION) or not np.isfinite(query_vector).all():
        raise ValueError("問題向量的維度或數值不正確。")
    if np.linalg.norm(query_vector) <= 0:
        raise ValueError("問題產生零向量。")
    faiss.normalize_L2(query_vector)
    return query_vector


def search_documents(question, top_k=3):
    if isinstance(top_k, bool) or not isinstance(top_k, (int, np.integer)) or top_k < 1:
        raise ValueError("top_k 必須是大於0的整數。")
    query_vector = embed_question(question)
    scores, positions = index.search(query_vector, min(int(top_k), index.ntotal))
    results = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        doc_id = index_doc_ids[int(position)]
        results.append({"rank": rank, "score": float(score), **documents_by_id[doc_id]})
    return results

## 9. 測試檢索結果
先問「Bagging 和隨機森林有什麼差別？」，閱讀取回的正文及來源頁碼。
可比較 top-k 為 1、3、5；相似度只用於排序，不代表答案正確率。

In [ ]:
QUESTION = "Bagging 和隨機森林有什麼差別？"
TOP_K = 3
results = search_documents(QUESTION, TOP_K)

display(pd.DataFrame([
    {
        "排名": r["rank"],
        "doc_id": r["doc_id"],
        "標題": r["metadata"]["title"],
        "相似度": r["score"],
        "來源文件": r["metadata"].get("source"),
        "頁碼": r["metadata"].get("pages"),
    }
    for r in results
]))

for result in results:
    metadata = result["metadata"]
    print(f'\n[{result["doc_id"]}] {metadata["title"]} ({result["score"]:.4f})')
    print(result["text"])
    print("來源：", metadata.get("source"), "｜頁碼：", metadata.get("pages"))


## 10. 保存知識庫
保存 `index.faiss`、`documents.json`、`index_doc_ids.json` 與 `config.json`。
這四個檔案一起交給 RAG02；設定檔記錄模型版本與檔案雜湊，不包含模型權重或 API 金鑰。

In [14]:
EXPORT_DIR = Path(tempfile.mkdtemp(prefix="rag_index_"))
faiss.write_index(index, str(EXPORT_DIR / "index.faiss"))

for filename, data in [("documents.json", documents), ("index_doc_ids.json", index_doc_ids)]:
    (EXPORT_DIR / filename).write_text(
        json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
    )

config = {
    "format_version": "1.0",
    "created_at": datetime.now(ZoneInfo("Asia/Taipei")).isoformat(timespec="seconds"),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "model_config_kwargs": {"vision_config": None, "audio_config": None},
    "dimension": DIMENSION,
    "normalize_embeddings": True,
    "index_type": "IndexFlatIP",
    "metric": "cosine_similarity_via_normalized_inner_product",
    "document_template": DOCUMENT_TEMPLATE,
    "query_template": QUERY_TEMPLATE,
    "encode_prompt": "",
    "max_seq_length": TOKEN_LIMIT,
    "document_count": len(documents),
    "document_unit": "one_document_one_vector",
    "document_text_includes_title": True,
    "package_versions": PACKAGE_VERSIONS,
    "build_device": DEVICE,
    "build_dtype": str(DTYPE),
    "source_archive": archive_name,
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "file_sha256": {
        name: hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
        for name in ["index.faiss", "documents.json", "index_doc_ids.json"]
    },
}
(EXPORT_DIR / "config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("已保存：")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, f"({path.stat().st_size:,} bytes)")
print("\n模型版本及前綴已寫入 config.json。")

已保存：
config.json (1,415 bytes)
documents.json (34,667 bytes)
index.faiss (95,277 bytes)
index_doc_ids.json (437 bytes)

模型版本及前綴已寫入 config.json。


## 11. 重新載入檢查
重新讀取保存的檔案，確認雜湊、區塊編號、維度與搜尋結果一致。
再和 NumPy 內積排序比較；這能檢查儲存與計算，不能保證找回的內容一定能回答問題。

In [15]:
loaded_config = json.loads((EXPORT_DIR / "config.json").read_text(encoding="utf-8"))
for name, expected_hash in loaded_config["file_sha256"].items():
    actual_hash = hashlib.sha256((EXPORT_DIR / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, f"檔案完整性檢查失敗：{name}"

loaded_index = faiss.read_index(str(EXPORT_DIR / "index.faiss"))
loaded_documents = json.loads((EXPORT_DIR / "documents.json").read_text(encoding="utf-8"))
loaded_ids = json.loads((EXPORT_DIR / "index_doc_ids.json").read_text(encoding="utf-8"))
assert loaded_index.ntotal == len(loaded_documents) == len(loaded_ids) == loaded_config["document_count"]
assert loaded_index.d == loaded_config["dimension"]
assert len(set(loaded_ids)) == len(loaded_ids)
assert set(loaded_ids) == {doc["doc_id"] for doc in loaded_documents}
assert loaded_documents == documents and loaded_ids == index_doc_ids

verification_query = embed_question(QUESTION)
k = min(TOP_K, index.ntotal)
before_scores, before_positions = index.search(verification_query, k)
after_scores, after_positions = loaded_index.search(verification_query, k)
np.testing.assert_array_equal(before_positions, after_positions)
np.testing.assert_allclose(before_scores, after_scores, atol=1e-6, rtol=1e-6)

all_scores = (embeddings @ verification_query[0]).ravel()
expected_top_scores = np.sort(all_scores)[::-1][:k]
np.testing.assert_allclose(after_scores[0], expected_top_scores, atol=1e-5, rtol=1e-5)
np.testing.assert_allclose(after_scores[0], all_scores[after_positions[0]], atol=1e-5, rtol=1e-5)
print("通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。")

通過：檔案完整性、文件對照、索引維度、保存前後一致，以及精確搜尋檢查。


## 12. 匯出 ZIP
將四個知識庫檔案打包成 `ml_week5_vectorDB.zip`，直接存進本機作業資料夾。
資料或 embedding 設定改變時，需要重新建庫與匯出，供 RAG02 載入。

In [17]:
# 直接儲存到本機作業資料夾
EXPORT_ZIP = (
    Path(r"C:\Users\LeePin\Desktop\個人\碩士班\生成式AI課\作業七")
    / EXPORT_ZIP_NAME
)

required_files = [
    "index.faiss",
    "documents.json",
    "index_doc_ids.json",
    "config.json",
]

with zipfile.ZipFile(
    EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for name in required_files:
        archive.write(EXPORT_DIR / name, arcname=name)

# 檢查壓縮檔完整性與內容
with zipfile.ZipFile(EXPORT_ZIP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == set(required_files)

print("已儲存：", EXPORT_ZIP)
print("大小：", f"{EXPORT_ZIP.stat().st_size:,} bytes")

已儲存： C:\Users\LeePin\Desktop\個人\碩士班\生成式AI課\作業七\ml_week5_vectorDB.zip
大小： 73,383 bytes


## 13. 本次實作整理
改用 JSON 保存資料，依主題切塊並保留來源頁碼；以 RTX 5060 建立 31 個區塊的向量索引。
RAG01 完成建庫與檢索，下一步到 RAG02 接上 LLM，測試有依據回答與資料不足的情況。

## 參考資料
[EmbeddingGemma 2 官方指引](https://ai.google.dev/gemma/docs/embeddinggemma/inference-embeddinggemma-with-sentence-transformers)｜[模型頁面](https://huggingface.co/google/embeddinggemma-2)｜[FAISS 入門](https://github.com/facebookresearch/faiss/wiki/Getting-started)